<a href="https://www.kaggle.com/code/lamontesmith/zero-shot-to-lora-optimizing-fine-tuning-agent?scriptVersionId=309523066" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Week 7: Optimizing & Fine-Tuning Agents
## Customer Support Ticket Classifier — From Zero-Shot to Fine-Tuned

**Author:** Lamonte Smith  
**Program:** Interview Kickstart Applied Agentic AI  
**GitHub:** [github.com/LSmithPMP/ik-agentic-ai-assignments](https://github.com/LSmithPMP/ik-agentic-ai-assignments)  
**Date:** April 2026

---

## Overview

Build the same customer support ticket classifier four different ways and benchmark each approach against accuracy, cost, and latency — generating real data to drive build-vs-buy decisions.

| Stage | Approach | Model | What You Learn |
|-------|----------|-------|----------------|
| 1 | Zero-Shot | GPT-4o-mini | Baseline — cheapest, no examples |
| 2 | Few-Shot | GPT-4o-mini | Does adding examples help? |
| 3 | Few-Shot | GPT-4o | Is a bigger model worth the cost? |
| 4 | LoRA Fine-Tune | SmolLM2-1.7B | Can a tiny self-hosted model compete? |
| 5 | Compare All | — | Cost vs quality vs latency tradeoffs |

## Hardware Required
**Enable GPU:** Settings → Accelerator → GPU T4 x2 (free on Kaggle)

## Results Preview

| Approach | Accuracy | Time | Cost/1K |
|----------|----------|------|---------|
| Zero-Shot GPT-4o-mini | 92.9% | 12.5s | $0.02 |
| Few-Shot GPT-4o-mini | 92.9% | 7.9s | $0.03 |
| Few-Shot GPT-4o | **100%** | 11.6s | $0.40 |
| LoRA SmolLM2-1.7B | 78.6% | 27.7s | ~$0 |

## Stage 0: Environment Setup

In [ ]:
!pip install -q openai transformers datasets peft accelerate bitsandbytes trl torch scikit-learn

## Configure API Keys

**Kaggle setup:**
1. Go to **Add-ons → Secrets** in the Kaggle notebook toolbar
2. Add `OPENAI_API_KEY` with your OpenAI key
3. Toggle access on

In [ ]:
import os
from openai import OpenAI

try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("OPENAI_API_KEY")
    print("API key loaded from Kaggle Secrets")
except Exception:
    api_key = input("Enter your OpenAI API key: ")
    print("API key set manually")

client = OpenAI(api_key=api_key)
print("OpenAI client ready")

## The Dataset: 66 Customer Support Tickets

In [ ]:
import random
random.seed(42)

tickets = [
    # BILLING (17 tickets)
    ("I was charged twice for my subscription this month", "billing"),
    ("Can I get a refund for the unused portion of my plan?", "billing"),
    ("My credit card was declined but I have sufficient funds", "billing"),
    ("I need to update my payment method to a new card", "billing"),
    ("The price increased without any notification", "billing"),
    ("I see an unauthorized charge on my account", "billing"),
    ("Can you explain the charges on my latest invoice?", "billing"),
    ("I cancelled but was still charged for next month", "billing"),
    ("Is there a discount for annual billing?", "billing"),
    ("My promo code isn't applying the discount correctly", "billing"),
    ("I need a receipt for my last three payments", "billing"),
    ("Why was I charged a late fee?", "billing"),
    ("Can I split my payment across two credit cards?", "billing"),
    ("The auto-renewal charged the wrong amount", "billing"),
    ("I need to dispute a charge from last month", "billing"),
    ("When will my refund be processed?", "billing"),
    ("I was charged sales tax but my state is tax-exempt", "billing"),
    # TECHNICAL (17 tickets)
    ("The app crashes every time I try to upload a file", "technical"),
    ("I'm getting a 500 error when accessing the dashboard", "technical"),
    ("The search function returns no results even for exact matches", "technical"),
    ("My data export is missing several columns", "technical"),
    ("The mobile app won't sync with the desktop version", "technical"),
    ("Page load times have been extremely slow this week", "technical"),
    ("The API is returning malformed JSON responses", "technical"),
    ("I can't connect my third-party integration", "technical"),
    ("The notification system isn't sending email alerts", "technical"),
    ("Two-factor authentication codes aren't being accepted", "technical"),
    ("The file preview feature shows a blank screen", "technical"),
    ("My scheduled reports aren't generating automatically", "technical"),
    ("The drag-and-drop feature stopped working after the update", "technical"),
    ("I'm getting timeout errors on large database queries", "technical"),
    ("The webhook endpoint isn't receiving any payloads", "technical"),
    ("Charts and graphs aren't rendering in the analytics tab", "technical"),
    ("The bulk import tool fails silently with no error message", "technical"),
    # ACCOUNT (16 tickets)
    ("I forgot my password and the reset email never arrives", "account"),
    ("I need to change the email address on my account", "account"),
    ("How do I add another user to my team account?", "account"),
    ("My account was locked after too many login attempts", "account"),
    ("I want to upgrade from the free plan to professional", "account"),
    ("Can I transfer ownership of my account to a colleague?", "account"),
    ("I need to delete my account and all associated data", "account"),
    ("How do I enable single sign-on for my organization?", "account"),
    ("My profile picture won't update", "account"),
    ("I can't access the admin panel even though I'm an admin", "account"),
    ("How do I merge two duplicate accounts?", "account"),
    ("I need to change my username", "account"),
    ("Can I temporarily deactivate my account?", "account"),
    ("My account permissions seem wrong after the migration", "account"),
    ("How do I set up role-based access for my team?", "account"),
    ("I need to export all my account data before closing it", "account"),
    # SHIPPING (16 tickets)
    ("My order hasn't arrived and it's been two weeks", "shipping"),
    ("The tracking number shows no movement for 5 days", "shipping"),
    ("I received the wrong item in my package", "shipping"),
    ("Can I change my delivery address after placing the order?", "shipping"),
    ("The package arrived damaged", "shipping"),
    ("Do you offer express shipping options?", "shipping"),
    ("I need to return an item — how do I get a shipping label?", "shipping"),
    ("My order was marked delivered but I never received it", "shipping"),
    ("Can I pick up my order from a local facility instead?", "shipping"),
    ("The shipping cost seems too high for a small item", "shipping"),
    ("I need to ship to an international address", "shipping"),
    ("My package was returned to sender — why?", "shipping"),
    ("How long does standard shipping take?", "shipping"),
    ("I want to consolidate multiple orders into one shipment", "shipping"),
    ("The delivery driver left my package in the rain", "shipping"),
    ("Can I schedule a specific delivery time window?", "shipping"),
]

random.shuffle(tickets)
split_idx = int(len(tickets) * 0.8)
train_data = tickets[:split_idx]
test_data = tickets[split_idx:]

print(f"Total tickets: {len(tickets)}")
print(f"Training set:  {len(train_data)} tickets")
print(f"Test set:      {len(test_data)} tickets")
print(f"Categories: {sorted(set(cat for _, cat in tickets))}")

## Helper Functions

In [ ]:
import time

CATEGORIES = ["billing", "technical", "account", "shipping"]

def classify_openai(ticket_text, model="gpt-4o-mini", few_shot_examples=None):
    system_prompt = (
        "You are a customer support ticket classifier. "
        "Classify each ticket into exactly one category: billing, technical, account, or shipping. "
        "Respond with ONLY the category name, nothing else."
    )
    messages = [{"role": "system", "content": system_prompt}]
    if few_shot_examples:
        for example_text, example_label in few_shot_examples:
            messages.append({"role": "user", "content": example_text})
            messages.append({"role": "assistant", "content": example_label})
    messages.append({"role": "user", "content": ticket_text})
    response = client.chat.completions.create(
        model=model, messages=messages, temperature=0, max_tokens=10
    )
    return response.choices[0].message.content.strip().lower()


def evaluate(predictions, actuals):
    correct = sum(p == a for p, a in zip(predictions, actuals))
    accuracy = correct / len(actuals)
    print(f"\n{'='*50}")
    print(f"Overall Accuracy: {accuracy:.1%} ({correct}/{len(actuals)})")
    print(f"{'='*50}")
    for cat in CATEGORIES:
        cat_indices = [i for i, a in enumerate(actuals) if a == cat]
        if cat_indices:
            cat_correct = sum(predictions[i] == actuals[i] for i in cat_indices)
            print(f"  {cat:12s}: {cat_correct}/{len(cat_indices)} correct")
    errors = [(actuals[i], predictions[i], i) for i in range(len(actuals)) if predictions[i] != actuals[i]]
    if errors:
        print(f"\nMisclassifications ({len(errors)}):")
        for actual, predicted, idx in errors[:5]:
            print(f"  \"{test_data[idx][0][:60]}...\"")
            print(f"    Expected: {actual} -> Got: {predicted}")
    return accuracy

print("Helper functions ready")

## Stage 1: Zero-Shot GPT-4o-mini

In [ ]:
print("Stage 1: Zero-Shot GPT-4o-mini")
print("=" * 50)

start = time.time()
predictions_zeroshot = [classify_openai(t, model="gpt-4o-mini") for t, _ in test_data]
elapsed_zeroshot = time.time() - start
actuals = [label for _, label in test_data]

acc_zeroshot = evaluate(predictions_zeroshot, actuals)
print(f"\nTime: {elapsed_zeroshot:.1f}s | Model: gpt-4o-mini | Cost/1K: $0.02")

## Stage 2: Few-Shot GPT-4o-mini

In [ ]:
few_shot_examples = []
seen_categories = set()
for text, label in train_data:
    if label not in seen_categories:
        few_shot_examples.append((text, label))
        seen_categories.add(label)
    if len(seen_categories) == 4:
        break

print("Few-shot examples:")
for text, label in few_shot_examples:
    print(f"  [{label:10s}] \"{text[:60]}...\"")

start = time.time()
predictions_fewshot_mini = [classify_openai(t, model="gpt-4o-mini", few_shot_examples=few_shot_examples) for t, _ in test_data]
elapsed_fewshot_mini = time.time() - start

print("\nStage 2: Few-Shot GPT-4o-mini")
acc_fewshot_mini = evaluate(predictions_fewshot_mini, actuals)
print(f"\nTime: {elapsed_fewshot_mini:.1f}s | Model: gpt-4o-mini + 4 examples | Cost/1K: $0.03")

## Stage 3: Few-Shot GPT-4o

In [ ]:
print("Stage 3: Few-Shot GPT-4o")
print("=" * 50)

start = time.time()
predictions_fewshot_4o = [classify_openai(t, model="gpt-4o", few_shot_examples=few_shot_examples) for t, _ in test_data]
elapsed_fewshot_4o = time.time() - start

acc_fewshot_4o = evaluate(predictions_fewshot_4o, actuals)
print(f"\nTime: {elapsed_fewshot_4o:.1f}s | Model: gpt-4o + 4 examples | Cost/1K: $0.40")

## Stage 4: LoRA Fine-Tuning (SmolLM2-1.7B)

**Why this matters:**
- No API costs at inference time
- Data stays on your infrastructure  
- Model is yours to deploy anywhere
- LoRA trains only ~0.37% of parameters

**Expected training time:** 3-5 minutes on T4 GPU

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

print(f"Model loaded!")
print(f"   Parameters: {model.num_parameters():,}")
print(f"   GPU Memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable parameters: {trainable:,} / {total:,}")
print(f"Trainable percentage: {100 * trainable / total:.2f}%")
print(f"LoRA trains only {100 * trainable / total:.2f}% of the model!")

In [ ]:
from datasets import Dataset

def format_for_training(ticket_text, label):
    return (
        f"Classify this customer support ticket into one of these categories: "
        f"billing, technical, account, shipping.\n\n"
        f"Ticket: {ticket_text}\n\nCategory: {label}"
    )

train_texts = [format_for_training(text, label) for text, label in train_data]
train_dataset = Dataset.from_dict({"text": train_texts})

print(f"Training examples: {len(train_dataset)}")
print(f"\nSample:\n{train_texts[0]}")

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="./lora-support-classifier",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=5,
    save_strategy="no",
    fp16=False,
    bf16=False,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    processing_class=tokenizer,
)

print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete!")

In [ ]:
def classify_finetuned(ticket_text):
    prompt = (
        f"Classify this customer support ticket into one of these categories: "
        f"billing, technical, account, shipping.\n\n"
        f"Ticket: {ticket_text}\n\nCategory:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=10, temperature=0.1, do_sample=False)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    result = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
    for cat in CATEGORIES:
        if cat in result:
            return cat
    return result.split()[0] if result else "unknown"


print("Stage 4: LoRA Fine-Tuned SmolLM2-1.7B")
print("=" * 50)

start = time.time()
predictions_lora = [classify_finetuned(t) for t, _ in test_data]
elapsed_lora = time.time() - start

acc_lora = evaluate(predictions_lora, actuals)
print(f"\nTime: {elapsed_lora:.1f}s | Model: SmolLM2-1.7B + LoRA | Cost/1K: ~$0 (self-hosted)")

## Stage 5: The Comparison

In [ ]:
results = {
    "Zero-Shot GPT-4o-mini": {"accuracy": acc_zeroshot, "time": elapsed_zeroshot, "cost_per_1k": "$0.02", "setup_effort": "None"},
    "Few-Shot GPT-4o-mini": {"accuracy": acc_fewshot_mini, "time": elapsed_fewshot_mini, "cost_per_1k": "$0.03", "setup_effort": "~1 hour"},
    "Few-Shot GPT-4o": {"accuracy": acc_fewshot_4o, "time": elapsed_fewshot_4o, "cost_per_1k": "$0.40", "setup_effort": "~1 hour"},
    "LoRA SmolLM2-1.7B": {"accuracy": acc_lora, "time": elapsed_lora, "cost_per_1k": "~$0 (self-hosted)", "setup_effort": "~1 week"},
}

print("\n" + "=" * 75)
print(f"{'Approach':<25} {'Accuracy':>10} {'Time':>10} {'Cost/1K':>15} {'Setup':>12}")
print("=" * 75)
for name, r in results.items():
    print(f"{name:<25} {r['accuracy']:>9.1%} {r['time']:>9.1f}s {r['cost_per_1k']:>15} {r['setup_effort']:>12}")
print("=" * 75)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
names = list(results.keys())
accuracies = [results[n]["accuracy"] for n in names]
times = [results[n]["time"] for n in names]
colors = ["#3B82F6", "#06B6D4", "#8B5CF6", "#F59E0B"]

axes[0].barh(names, accuracies, color=colors, height=0.6)
axes[0].set_xlim(0, 1.05)
axes[0].set_xlabel("Accuracy")
axes[0].set_title("Accuracy by Approach", fontweight="bold")
for i, v in enumerate(accuracies):
    axes[0].text(v + 0.01, i, f"{v:.1%}", va="center", fontweight="bold")

axes[1].barh(names, times, color=colors, height=0.6)
axes[1].set_xlabel("Time (seconds)")
axes[1].set_title("Inference Time (test set)", fontweight="bold")
for i, v in enumerate(times):
    axes[1].text(v + 0.2, i, f"{v:.1f}s", va="center", fontweight="bold")

plt.tight_layout()
plt.savefig("comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved as comparison.png")

In [ ]:
model.save_pretrained("./lora-adapter")

import os
adapter_size = sum(
    os.path.getsize(os.path.join("./lora-adapter", f))
    for f in os.listdir("./lora-adapter")
    if f.endswith((".bin", ".safetensors"))
)
full_model_size = 1.7e9 * 2

print(f"Full model size:   ~{full_model_size/1e9:.1f} GB")
print(f"LoRA adapter size: {adapter_size/1e6:.1f} MB")
print(f"Compression ratio: {full_model_size/adapter_size:.0f}x smaller")
print(f"\nYou ship the {adapter_size/1e6:.1f}MB adapter, not the {full_model_size/1e9:.1f}GB model!")
print(f"The base model is shared across all your fine-tuned variants.")

## Key Takeaway

There is no universally best approach. The right choice depends on volume, budget, accuracy requirements, and data sensitivity.

| Scenario | Recommendation | Rationale |
|----------|---------------|----------|
| Startup, 100 tickets/day | Few-Shot GPT-4o-mini | Fine-tuning has a 3-year break-even at this volume |
| Enterprise, 100K tickets/day | Fine-tuned 7B+ open-source | GPT-4o costs $40K/day at this volume |
| Healthcare, any volume | Self-hosted fine-tuning | HIPAA blocks any external API — data must stay internal |

**The decision order:** Prompt engineering first → Model scaling only if quality demands it → Fine-tuning only when volume and data privacy justify the organizational investment.

---
*Lamonte Smith · [github.com/LSmithPMP](https://github.com/LSmithPMP) · Interview Kickstart Applied Agentic AI*